### Model free RL

Since we do not know about the environment dynamics, we try to estimate the Q and V values of different states by trial and errors.

#### Monte Carlo Learning

$$V(s_k) = E(r_k + \gamma V(s_{k+1})) $$

We estimate the V value by simple update term as follows:

$$R_{\sum} = \sum_{k=1}^{n}\gamma^{k}r_k$$
$$V^{new}(s_k) = V^{old}(s_k) + \frac{1}{n} (R_{\sum} - V^{old}(s_k))$$
$$Q^{new}(s_k,a_k) = Q^{old}(s_k,a_k) + \frac{1}{n} (R_{\sum} - Q^{old}(s_k,a_k))$$

If we reach the optimal value of $V(s_k)$, the difference term $(R_{\sum} - V^{old}(s_k)$ becomes 0 and $V(s_k)$ stops updating. But this gives equal weightage to all the steps hence, this is actually slow and we use _Temporal Difference_ which gives weightage to results nearer to the rewards. 

#### Temporal Difference 

$$V^{new}(s_k) = V^{old}(s_k) + \alpha (r_k + \gamma V^{old}(s_{k+1}) - V^{old}(s_k))$$
$$R_{\sum} = r_k + \gamma V^{old}(s_{k+1})$$

$R_{\sum}$ : TD Target Estimate

This is kind of 1$\Delta t$ delay between an action and reward and things correlated with 1$\Delta t$ will get strengthened in this TD(0) framework. 

$$V(s_k) = E(r_k + \gamma r_{k+1} + \gamma^2 V(s_{k+1})) $$
Therefore TD(1),
$$V^{new}(s_k) = V^{old}(s_k) + \alpha (r_k + \gamma r_{k+1} + \gamma^2 V^{old}(s_{k+1}) - V^{old}(s_k))$$
Here we are considering 2 steps into the future. This way we can go on to generate TD(N) famework. If n $\to$ $\infty$, this will converge to Monte Carlo Learning.

##### On Policy TD - SARSA

State-Action-Reward-State-Action
$$Q^{new}(s_k,a_k) = Q^{old}(s_k,a_k) + \alpha (r_k + \gamma Q^{old}(s_{k+1},a_{k+1}) - Q^{old}(s_k,a_k))$$

Sarsa is generally safer as it always follows what it thinks is best. Eg teaching a teenager to drive, here random actions can lead to accidents and hence Sarsa is preferred.

In [14]:
import gym
import random
import pandas as pd

env = gym.make('FrozenLake-v1')
def random_policy():
    return env.action_space.sample()

In [15]:
V = {}
for s in range(env.observation_space.n):
    V[s] = 0.0
    
Q = {}
for s in range(env.observation_space.n):
    for a in range(env.action_space.n):
        Q[(s,a)] = 0.0

In [16]:
alpha = 0.85
gamma = 0.90
num_episodes = 50000
num_timesteps = 1000
for i in range(num_episodes):
    s = env.reset()
    for t in range(num_timesteps):
        a = random_policy()
        s_, r, done, _ = env.step(a)
        V[s] += alpha * (r + gamma*V[s_] - V[s]) ## TD(0)
        s = s_
        if done:
            break

In [17]:
def epsilon_greedy(state, epsilon):
    if random.uniform(0,1) < epsilon:
        return env.action_space.sample()
    else:
        return max(list(range(env.action_space.n)), key = lambda x: Q[(state, x)])

In [18]:
alpha = 0.85
gamma = 0.90
epsilon = 0.80
num_episodes = 50000
num_timesteps = 1000
for i in range(num_episodes):
    s = env.reset()
    a = epsilon_greedy(s,epsilon)
    for t in range(num_timesteps):
        s_, r, done, _ = env.step(a)
        a_ = epsilon_greedy(s,epsilon)
        Q[(s,a)] += alpha * (r + gamma*Q[(s_,a_)] - Q[(s,a)]) ## SARSA
        a = a_
        s = s_
        if done:
            break

In [19]:
df = pd.DataFrame(list(V.items()), columns=['state', 'value'])
df

,state,value
0,0,0.000549
1,1,0.000002
2,2,0.000091
3,3,0.000027
4,4,0.003360
5,5,0.000000
6,6,0.000020
7,7,0.000000
8,8,0.000520
9,9,0.511335


##### Off Policy TD  - Q learning

Q learning is just Temporal Difference Learning but on Q Function

$$Q^{new}(s_k,a_k) = Q^{old}(s_k,a_k) + \alpha (r_k + \gamma \mathop{max}_{\textbf{a}} Q^{old}(s_{k+1},a) - Q^{old}(s_k,a_k))$$

We can compute any random action to get $r_k$ but while computing $Q^{new}(s_k,a_k)$, we maximize over *a*. Here is the updated line in the above code.

`a_ = np.argmax([Q[(s_, a)] for a in range(env.action_space.n)])`

Q Learning is faster and can learn from immitation and experience replay. \
Different strategies exist for introducing this randomness : epsilon greedy, softmax exploration, upper confidence bound, thompson sampling etc